# Deep Learning: LSTM

试试用LSTM来做时间序列预测，看看效果怎么样。

思路：
- 用MinMaxScaler归一化
- 滑动窗口构造特征（past 12 months -> predict next month）
- 递归预测测试集

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_raw_data, get_monthly_sales, train_test_split_ts
from src.models.lstm_model import LSTMForecaster
from src.evaluate import evaluate_forecast, compare_models
from src.utils import plot_forecast, set_seed

set_seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
df = load_raw_data('../data/raw/train.csv')
monthly = get_monthly_sales(df)
train, test = train_test_split_ts(monthly, test_year=2018)
print(f'Train: {len(train)}, Test: {len(test)}')

## 1. LSTM Model

先用默认参数跑一次看看

In [ ]:
lstm = LSTMForecaster(
    seq_len=12,
    hidden_size=64,
    num_layers=2,
    lr=0.001,
    epochs=150,
    batch_size=8,
)
lstm.fit(train)

In [ ]:
# training loss
fig = lstm.plot_loss()
plt.show()

In [ ]:
lstm_pred = lstm.predict(steps=len(test), test_index=test.index)

result = evaluate_forecast(test, lstm_pred, 'LSTM')
print(result)

In [ ]:
fig = plot_forecast(train, test, lstm_pred, 'LSTM Forecast (seq_len=12, hidden=64)')
plt.savefig('../results/figures/lstm_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 调一下超参数

试试不同的hidden_size和seq_len

In [ ]:
# TODO: could do a proper grid search here but the dataset is small
# so let's just try a few combos manually

configs = [
    {'seq_len': 6, 'hidden_size': 32, 'epochs': 150},
    {'seq_len': 12, 'hidden_size': 64, 'epochs': 150},
    {'seq_len': 12, 'hidden_size': 128, 'epochs': 200},
]

lstm_results = []
for i, cfg in enumerate(configs):
    print(f'\n--- Config {i+1}: {cfg} ---')
    set_seed(42)
    model = LSTMForecaster(
        seq_len=cfg['seq_len'],
        hidden_size=cfg['hidden_size'],
        num_layers=2,
        lr=0.001,
        epochs=cfg['epochs'],
        batch_size=8,
    )
    model.fit(train)
    pred = model.predict(steps=len(test), test_index=test.index)
    name = f"LSTM(seq={cfg['seq_len']}, h={cfg['hidden_size']})"
    res = evaluate_forecast(test, pred, name)
    lstm_results.append(res)
    print(f'  -> {res}')

In [ ]:
lstm_comparison = compare_models(lstm_results)
print(lstm_comparison.to_string())

## 3. 和统计模型对比

把之前notebook里跑的baseline结果拿过来一起比较

In [ ]:
# load previous results if available
import os
if os.path.exists('../results/model_comparison.csv'):
    prev = pd.read_csv('../results/model_comparison.csv')
    print('Previous results:')
    print(prev.to_string(index=False))
else:
    print('No previous results found. Run 02_baseline_models.ipynb first.')
    print('Or run: python run_experiment.py')

## Summary

LSTM在这个小数据集上其实不一定比统计模型好，主要原因：
1. 数据量太少（只有48个月的训练数据），深度学习很难学到好的pattern
2. 递归预测误差会累积
3. 月度数据的季节性比较规律，统计模型处理这种case本来就很强

不过作为练习，LSTM的pipeline是完整的，可以直接应用到更大的数据集上。